# 04. Random Forest Model Training — Baseline (no preprocessing)

This notebook trains a **Random Forest** classifier with **default settings** on all **103 signal features**, without any preprocessing, to get a first baseline score.

It uses exactly the same data and steps as `04d_lightgbm_baseline.ipynb` and `04a_logistic_regression_baseline.ipynb`, so the results can be compared.

Why no preprocessing is acceptable here:
- **Scaling** is not needed: Random Forest is made of decision trees, which split on value thresholds, so feature scale does not matter.
- **Imputation** is not needed: the 103 features have no missing values.
- **Feature selection** is not done: this is the *full feature set* baseline.

Rules followed:
- Train on `train.csv` only, score on `val.csv` only.
- `test.csv` is **not loaded** in this notebook.
- `record_id`, `patient_id` and `figo_label` are not used as input features.
- Positive class = abnormal (`figo_label = 1`). Threshold = 0.5 (default).

## Step 1: Setup and imports

In [1]:
import sys
sys.path.append('../..')

import time
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score, confusion_matrix

from config import config
from src.data_preprocessing import get_feature_columns

## Step 2: Load the training and validation sets

The test set stays untouched until the final evaluation.

In [2]:
# Load train and validation sets only (test set stays untouched)
train_df = pd.read_csv(config.PROCESSED_TRAIN_FILE)
val_df = pd.read_csv(config.PROCESSED_VAL_FILE)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

Train shape: (14540, 106)
Validation shape: (3115, 106)


## Step 3: Separate features (X) and target (y)

In [3]:
# Get the 103 signal features (IDs and target are excluded)
feature_cols = get_feature_columns(train_df)

X_train = train_df[feature_cols]
y_train = train_df[config.TARGET_COLUMN]

X_val = val_df[feature_cols]
y_val = val_df[config.TARGET_COLUMN]

print("Train:", X_train.shape, " Validation:", X_val.shape)

Number of feature columns found: 103
Train: (14540, 103)  Validation: (3115, 103)


## Step 4: Create and train the Random Forest model

Default hyperparameters (100 trees, no depth limit); only the random seed is fixed for reproducibility.

In [4]:
# Create Random Forest with default settings (only the seed is fixed)
model = RandomForestClassifier(random_state=config.RANDOM_STATE)

# Train on the training set only
start_time = time.time()
model.fit(X_train, y_train)
train_seconds = time.time() - start_time

print("Training time (seconds):", round(train_seconds, 2))
print("Number of trees:", len(model.estimators_))

Training time (seconds): 8.9
Number of trees: 100


## Step 5: Predict on the validation set

In [5]:
# Class predictions (threshold 0.5 by default)
val_pred = model.predict(X_val)

# Probability of abnormal (1), needed for ROC-AUC
val_prob = model.predict_proba(X_val)[:, 1]

## Step 6: Validation metrics

Positive class = abnormal (`figo_label = 1`).

In [6]:
print("Accuracy:         ", round(accuracy_score(y_val, val_pred), 4))
print("Precision:        ", round(precision_score(y_val, val_pred), 4))
print("Recall:           ", round(recall_score(y_val, val_pred), 4))
print("F1-score:         ", round(f1_score(y_val, val_pred), 4))
print("ROC-AUC:          ", round(roc_auc_score(y_val, val_prob), 4))
print("Balanced accuracy:", round(balanced_accuracy_score(y_val, val_pred), 4))

print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_val, val_pred))

Accuracy:          0.886
Precision:         0.886
Recall:            0.9045
F1-score:          0.8952
ROC-AUC:           0.9616
Balanced accuracy: 0.8845
Confusion matrix [[TN, FP], [FN, TP]]:
[[1244  195]
 [ 160 1516]]


## Step 7: Check for overfitting

If the training F1 is much higher than the validation F1, the model is overfitting.
Default Random Forest trees grow with no depth limit, so a training F1 close to 1.0 is expected.

In [7]:
# Training F1, compared with validation F1 from Step 6
train_pred = model.predict(X_train)
print("Training F1:  ", round(f1_score(y_train, train_pred), 4))
print("Validation F1:", round(f1_score(y_val, val_pred), 4))

Training F1:   1.0
Validation F1: 0.8952


## Step 8: Top 15 most important features

Random Forest gives each feature an importance score (how much it helped the trees separate normal from abnormal).
This is only a first look, learned from the training set. It is **not** the feature selection of notebook 03.

In [8]:
# Put the feature names and their importance scores into a table
importance_table = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_,
})

# Sort from most to least important
importance_table = importance_table.sort_values("importance", ascending=False)

print(importance_table.head(15).to_string(index=False))

     feature  importance
 FU_RE_WCTTF    0.051838
       F_D_N    0.049501
  FU_V_WCTTF    0.047108
        F_SD    0.044858
      F_VD_N    0.038209
 FU_SE_WCTTF    0.030250
  F_LPF_ST11    0.027344
  FU_M_WCTTF    0.027087
   U_RE_STTF    0.023661
    U_V_STTF    0.021857
FU_LPF_XWT11    0.020028
  F_LPF_ST10    0.019751
FU_LPF_XWT10    0.019323
FU_LPF_WCT11    0.018782
FU_LPF_WCT10    0.018102
